In [1]:
# ============================================================
# DATATHON 2026 — HALF-MONTH USER RETENTION / COHORT ANALYSIS
# ============================================================
# Ý tưởng:
# - Bucket nửa tháng:
#     H1 = ngày 01 -> 15
#     H2 = ngày 16 -> cuối tháng
# - Mỗi user có cohort = bucket đầu tiên user xuất hiện
# - Retention +k = user cohort đó có active lại ở bucket cohort + k hay không
#
# Output:
# - Retention heatmap
# - Average retention curve
# - Cohort size chart
# - Survival / return-after curve
# - Optional: city/category switching sau cohort
#
# Chạy 1 cell từ đầu tới cuối.
# ============================================================

import os
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# ============================================================
# 0. CONFIG
# ============================================================

# Sửa path này nếu dataset của bạn nằm chỗ khác
DATA_ROOT_CANDIDATES = [
    "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2",
    "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1",
    "/kaggle/input/datasets/luquang231/datathonvinu",
    "/kaggle/input/datathon2026-2",
    "/kaggle/input/datathon2026-1",
]

EVENTS_SUBDIR = "fact_user_events"

OUTPUT_DIR = "/kaggle/working/user_retention_half_month"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# login: đo retention user thật tốt hơn
# non-login: không khuyến nghị vì user_id đổi theo session
# all: nếu muốn gom cả login + non-login
LOGIN_MODE = "login"       # "login" | "non-login" | "all"

# Active definition:
# None = user active nếu có bất kỳ event nào trong bucket
# Hoặc set list event_type nếu chỉ muốn active = contact/pageview gì đó
ACTIVE_EVENT_TYPES = None
# Ví dụ:
# ACTIVE_EVENT_TYPES = ["pageview"]
# ACTIVE_EVENT_TYPES = ["view_phone", "contact_chat", "contact_zalo", "contact_sms"]

# Optional date filter. Để None thì auto dùng toàn bộ data.
START_DATE = None          # ví dụ "2024-09-01"
END_DATE = None            # ví dụ "2026-05-31"

# Nếu OOM, giảm threads xuống 1 hoặc 2
DUCKDB_MEMORY_LIMIT = "6GB"
DUCKDB_THREADS = 2
DUCKDB_MAX_TEMP_DIR_SIZE = "80GB"

# Switching analysis nặng hơn vì phải tính city/category mode theo user-bucket.
# Nếu bị OOM, đổi thành False.
DO_SWITCH_ANALYSIS = True

# Positive/contact event sets để thống kê thêm trong retention table
POSITIVE_EVENT_TYPES = [
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms"
]
STRICT_CONTACT_EVENT_TYPES = [
    "view_phone", "contact_chat", "contact_zalo", "contact_sms"
]

# ============================================================
# 1. SETUP
# ============================================================

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly"])
    import plotly.express as px
    import plotly.graph_objects as go


def find_events_path():
    for root in DATA_ROOT_CANDIDATES:
        p = os.path.join(root, EVENTS_SUBDIR)
        if os.path.exists(p):
            return p
    
    # fallback: scan /kaggle/input
    for base, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(base) == EVENTS_SUBDIR:
            return base
    
    raise FileNotFoundError(
        "Không tìm thấy fact_user_events. Hãy sửa DATA_ROOT_CANDIDATES hoặc EVENTS_SUBDIR."
    )


EVENTS_PATH = find_events_path()
EVENTS_GLOB = os.path.join(EVENTS_PATH, "*.parquet").replace("\\", "/")

print("EVENTS_PATH:", EVENTS_PATH)
print("EVENTS_GLOB:", EVENTS_GLOB)
print("OUTPUT_DIR :", OUTPUT_DIR)

duckdb_path = os.path.join(OUTPUT_DIR, "retention_half_month.duckdb")
tmp_dir = os.path.join(OUTPUT_DIR, "duckdb_tmp")
os.makedirs(tmp_dir, exist_ok=True)

if os.path.exists(duckdb_path):
    os.remove(duckdb_path)

con = duckdb.connect(duckdb_path)

con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA temp_directory='{tmp_dir}'")
con.execute(f"PRAGMA max_temp_directory_size='{DUCKDB_MAX_TEMP_DIR_SIZE}'")
con.execute("SET preserve_insertion_order=false")

# ============================================================
# 2. SQL FILTERS
# ============================================================

def sql_str_list(values):
    return ", ".join(["'" + str(v).replace("'", "''").lower() + "'" for v in values])

login_filter_sql = ""
if LOGIN_MODE == "login":
    login_filter_sql = "AND LOWER(TRIM(CAST(is_login AS VARCHAR))) = 'login'"
elif LOGIN_MODE == "non-login":
    login_filter_sql = "AND LOWER(TRIM(CAST(is_login AS VARCHAR))) = 'non-login'"
elif LOGIN_MODE == "all":
    login_filter_sql = ""
else:
    raise ValueError("LOGIN_MODE phải là 'login', 'non-login', hoặc 'all'.")

active_filter_sql = ""
if ACTIVE_EVENT_TYPES is not None:
    active_filter_sql = f"""
    AND LOWER(TRIM(CAST(event_type AS VARCHAR))) IN ({sql_str_list(ACTIVE_EVENT_TYPES)})
    """

date_filter_sql = ""
if START_DATE is not None:
    date_filter_sql += f" AND event_date >= DATE '{START_DATE}' "
if END_DATE is not None:
    date_filter_sql += f" AND event_date <= DATE '{END_DATE}' "

positive_sql = sql_str_list(POSITIVE_EVENT_TYPES)
strict_contact_sql = sql_str_list(STRICT_CONTACT_EVENT_TYPES)

# ============================================================
# 3. QUICK DATA RANGE CHECK
# ============================================================

range_query = f"""
WITH src AS (
    SELECT
        COALESCE(
            TRY_CAST(date AS DATE),
            CAST(event_ts AS DATE)
        ) AS event_date,
        LOWER(TRIM(CAST(is_login AS VARCHAR))) AS is_login_clean
    FROM read_parquet('{EVENTS_GLOB}')
    WHERE COALESCE(TRY_CAST(date AS DATE), CAST(event_ts AS DATE)) IS NOT NULL
)
SELECT
    MIN(event_date) AS min_date,
    MAX(event_date) AS max_date,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN is_login_clean = 'login' THEN 1 ELSE 0 END) AS login_rows,
    SUM(CASE WHEN is_login_clean = 'non-login' THEN 1 ELSE 0 END) AS non_login_rows
FROM src
"""

data_range_df = con.execute(range_query).df()
display(data_range_df)

data_range_df.to_csv(
    os.path.join(OUTPUT_DIR, "00_data_range_check.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 4. BUILD USER x HALF-MONTH BUCKET ACTIVITY
# ============================================================

print("\n[1/8] Building user_bucket_activity...")

create_user_bucket_activity_query = f"""
CREATE OR REPLACE TABLE user_bucket_activity AS
WITH src AS (
    SELECT
        CAST(user_id AS VARCHAR) AS user_id,

        COALESCE(
            TRY_CAST(date AS DATE),
            CAST(event_ts AS DATE)
        ) AS event_date,

        LOWER(TRIM(CAST(event_type AS VARCHAR))) AS event_type_clean,

        CASE
            WHEN city_name IS NULL OR TRIM(CAST(city_name AS VARCHAR)) = ''
            THEN NULL
            ELSE TRIM(CAST(city_name AS VARCHAR))
        END AS city_name_clean,

        TRY_CAST(category AS INTEGER) AS category_clean,

        LOWER(TRIM(CAST(is_login AS VARCHAR))) AS is_login_clean

    FROM read_parquet('{EVENTS_GLOB}')
    WHERE user_id IS NOT NULL
        {login_filter_sql}
        {active_filter_sql}
),
base AS (
    SELECT
        user_id,
        event_date,
        event_type_clean,
        city_name_clean,
        category_clean,

        CAST(
            EXTRACT(YEAR FROM event_date) * 24
            + (EXTRACT(MONTH FROM event_date) - 1) * 2
            + CASE
                WHEN EXTRACT(DAY FROM event_date) <= 15 THEN 0
                ELSE 1
              END
            AS BIGINT
        ) AS bucket_id

    FROM src
    WHERE event_date IS NOT NULL
        {date_filter_sql}
)
SELECT
    user_id,
    bucket_id,

    COUNT(*) AS events,
    SUM(CASE WHEN event_type_clean IN ({positive_sql}) THEN 1 ELSE 0 END) AS positive_events,
    SUM(CASE WHEN event_type_clean IN ({strict_contact_sql}) THEN 1 ELSE 0 END) AS strict_contact_events,

    COUNT(DISTINCT city_name_clean) AS distinct_cities_seen,
    COUNT(DISTINCT category_clean) AS distinct_categories_seen

FROM base
GROUP BY user_id, bucket_id
"""

con.execute(create_user_bucket_activity_query)

activity_summary_df = con.execute("""
SELECT
    COUNT(*) AS user_bucket_rows,
    COUNT(DISTINCT user_id) AS users,
    MIN(bucket_id) AS min_bucket_id,
    MAX(bucket_id) AS max_bucket_id,
    SUM(events) AS total_events,
    SUM(positive_events) AS total_positive_events,
    SUM(strict_contact_events) AS total_strict_contact_events
FROM user_bucket_activity
""").df()

display(activity_summary_df)
activity_summary_df.to_csv(
    os.path.join(OUTPUT_DIR, "01_user_bucket_activity_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 5. BUILD BUCKET CALENDAR
# ============================================================

print("\n[2/8] Building bucket calendar...")

con.execute("""
CREATE OR REPLACE TABLE bucket_calendar AS
WITH lim AS (
    SELECT
        MIN(bucket_id) AS min_bucket_id,
        MAX(bucket_id) AS max_bucket_id
    FROM user_bucket_activity
),
ids AS (
    SELECT r.bucket_id
    FROM lim
    CROSS JOIN LATERAL range(
        CAST(min_bucket_id AS BIGINT),
        CAST(max_bucket_id + 1 AS BIGINT)
    ) AS r(bucket_id)
),
parts AS (
    SELECT
        bucket_id,
        CAST(FLOOR(bucket_id / 24) AS INTEGER) AS yy,
        CAST(FLOOR((bucket_id % 24) / 2) + 1 AS INTEGER) AS mm,
        CAST(bucket_id % 2 AS INTEGER) AS half_idx
    FROM ids
),
starts AS (
    SELECT
        bucket_id,
        yy,
        mm,
        half_idx,
        CAST(
            CAST(yy AS VARCHAR)
            || '-'
            || LPAD(CAST(mm AS VARCHAR), 2, '0')
            || '-01'
            AS DATE
        )
        + CASE
            WHEN half_idx = 1 THEN INTERVAL '15 days'
            ELSE INTERVAL '0 days'
          END AS bucket_start
    FROM parts
)
SELECT
    bucket_id,
    CAST(bucket_start AS DATE) AS bucket_start,

    CASE
        WHEN half_idx = 0 THEN CAST(bucket_start + INTERVAL '14 days' AS DATE)
        ELSE CAST(DATE_TRUNC('month', bucket_start) + INTERVAL '1 month' - INTERVAL '1 day' AS DATE)
    END AS bucket_end,

    STRFTIME(bucket_start, '%Y-%m')
        || CASE WHEN half_idx = 0 THEN '-H1' ELSE '-H2' END AS bucket_label,

    half_idx
FROM starts
ORDER BY bucket_id
""")

bucket_calendar_df = con.execute("SELECT * FROM bucket_calendar ORDER BY bucket_id").df()
display(bucket_calendar_df)

bucket_calendar_df.to_csv(
    os.path.join(OUTPUT_DIR, "02_bucket_calendar.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 6. COHORT = FIRST ACTIVE BUCKET OF USER
# ============================================================

print("\n[3/8] Building user cohorts...")

con.execute("""
CREATE OR REPLACE TABLE user_cohort AS
SELECT
    user_id,
    MIN(bucket_id) AS cohort_bucket_id
FROM user_bucket_activity
GROUP BY user_id
""")

con.execute("""
CREATE OR REPLACE TABLE cohort_sizes AS
SELECT
    c.cohort_bucket_id,
    bc.bucket_label AS cohort_label,
    bc.bucket_start AS cohort_start,
    bc.bucket_end AS cohort_end,
    COUNT(*) AS cohort_users
FROM user_cohort c
LEFT JOIN bucket_calendar bc
    ON c.cohort_bucket_id = bc.bucket_id
GROUP BY
    c.cohort_bucket_id,
    bc.bucket_label,
    bc.bucket_start,
    bc.bucket_end
ORDER BY c.cohort_bucket_id
""")

cohort_sizes_df = con.execute("SELECT * FROM cohort_sizes ORDER BY cohort_bucket_id").df()
display(cohort_sizes_df)

cohort_sizes_df.to_csv(
    os.path.join(OUTPUT_DIR, "03_cohort_sizes.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 7. EXACT RETENTION MATRIX
# ============================================================

print("\n[4/8] Building exact retention table...")

con.execute("""
CREATE OR REPLACE TABLE retention_counts AS
SELECT
    c.cohort_bucket_id,
    uba.bucket_id - c.cohort_bucket_id AS bucket_offset,

    COUNT(*) AS active_users,
    SUM(uba.events) AS events,
    SUM(uba.positive_events) AS positive_events,
    SUM(uba.strict_contact_events) AS strict_contact_events,

    AVG(uba.events) AS mean_events_per_active_user,
    AVG(uba.positive_events) AS mean_positive_events_per_active_user,
    AVG(uba.strict_contact_events) AS mean_strict_contact_events_per_active_user

FROM user_bucket_activity uba
JOIN user_cohort c
    ON uba.user_id = c.user_id
WHERE uba.bucket_id >= c.cohort_bucket_id
GROUP BY
    c.cohort_bucket_id,
    uba.bucket_id - c.cohort_bucket_id
""")

con.execute("""
CREATE OR REPLACE TABLE retention_long AS
WITH maxb AS (
    SELECT MAX(bucket_id) AS max_bucket_id
    FROM bucket_calendar
),
grid AS (
    SELECT
        cs.cohort_bucket_id,
        r.bucket_offset
    FROM cohort_sizes cs
    CROSS JOIN maxb
    CROSS JOIN LATERAL range(
        0,
        CAST(maxb.max_bucket_id - cs.cohort_bucket_id + 1 AS BIGINT)
    ) AS r(bucket_offset)
)
SELECT
    g.cohort_bucket_id,
    bc0.bucket_label AS cohort_label,
    bc0.bucket_start AS cohort_start,
    bc0.bucket_end AS cohort_end,

    g.bucket_offset,
    '+' || CAST(g.bucket_offset AS VARCHAR) AS offset_label,

    g.cohort_bucket_id + g.bucket_offset AS active_bucket_id,
    bc1.bucket_label AS active_bucket_label,
    bc1.bucket_start AS active_bucket_start,
    bc1.bucket_end AS active_bucket_end,

    cs.cohort_users,

    COALESCE(rc.active_users, 0) AS active_users,
    COALESCE(rc.events, 0) AS events,
    COALESCE(rc.positive_events, 0) AS positive_events,
    COALESCE(rc.strict_contact_events, 0) AS strict_contact_events,

    COALESCE(rc.active_users, 0) * 1.0 / NULLIF(cs.cohort_users, 0) AS retention_rate,

    rc.mean_events_per_active_user,
    rc.mean_positive_events_per_active_user,
    rc.mean_strict_contact_events_per_active_user

FROM grid g
JOIN cohort_sizes cs
    ON g.cohort_bucket_id = cs.cohort_bucket_id
LEFT JOIN retention_counts rc
    ON g.cohort_bucket_id = rc.cohort_bucket_id
    AND g.bucket_offset = rc.bucket_offset
LEFT JOIN bucket_calendar bc0
    ON g.cohort_bucket_id = bc0.bucket_id
LEFT JOIN bucket_calendar bc1
    ON g.cohort_bucket_id + g.bucket_offset = bc1.bucket_id
ORDER BY
    g.cohort_bucket_id,
    g.bucket_offset
""")

retention_long_df = con.execute("SELECT * FROM retention_long").df()
display(retention_long_df.head(30))

retention_long_df.to_csv(
    os.path.join(OUTPUT_DIR, "04_retention_long_exact.csv"),
    index=False,
    encoding="utf-8-sig"
)

# Pivot matrix
retention_matrix_df = retention_long_df.pivot(
    index="cohort_label",
    columns="offset_label",
    values="retention_rate"
)

# Giữ thứ tự cohort đúng theo thời gian
cohort_order = cohort_sizes_df.sort_values("cohort_bucket_id")["cohort_label"].tolist()
offset_order = (
    retention_long_df[["bucket_offset", "offset_label"]]
    .drop_duplicates()
    .sort_values("bucket_offset")["offset_label"]
    .tolist()
)

retention_matrix_df = retention_matrix_df.reindex(index=cohort_order, columns=offset_order)

retention_matrix_df.to_csv(
    os.path.join(OUTPUT_DIR, "05_retention_matrix_exact.csv"),
    encoding="utf-8-sig"
)

display(retention_matrix_df)

# ============================================================
# 8. SURVIVAL / RETURN-AFTER RETENTION
# ============================================================
# exact retention +k = có active đúng ở bucket +k
# survival +k = user có quay lại ít nhất 1 lần ở bucket +k hoặc sau đó
# Với BĐS, survival thường hữu ích hơn vì user không nhất thiết active mỗi nửa tháng.

print("\n[5/8] Building survival / return-after table...")

con.execute("""
CREATE OR REPLACE TABLE user_max_offset AS
SELECT
    c.user_id,
    c.cohort_bucket_id,
    MAX(uba.bucket_id - c.cohort_bucket_id) AS max_observed_offset
FROM user_cohort c
JOIN user_bucket_activity uba
    ON c.user_id = uba.user_id
GROUP BY
    c.user_id,
    c.cohort_bucket_id
""")

con.execute("""
CREATE OR REPLACE TABLE max_offset_dist AS
SELECT
    cohort_bucket_id,
    max_observed_offset AS bucket_offset,
    COUNT(*) AS users_with_last_seen_here
FROM user_max_offset
GROUP BY
    cohort_bucket_id,
    max_observed_offset
""")

con.execute("""
CREATE OR REPLACE TABLE survival_long AS
WITH maxb AS (
    SELECT MAX(bucket_id) AS max_bucket_id
    FROM bucket_calendar
),
grid AS (
    SELECT
        cs.cohort_bucket_id,
        r.bucket_offset
    FROM cohort_sizes cs
    CROSS JOIN maxb
    CROSS JOIN LATERAL range(
        0,
        CAST(maxb.max_bucket_id - cs.cohort_bucket_id + 1 AS BIGINT)
    ) AS r(bucket_offset)
),
base AS (
    SELECT
        g.cohort_bucket_id,
        g.bucket_offset,
        COALESCE(d.users_with_last_seen_here, 0) AS users_with_last_seen_here
    FROM grid g
    LEFT JOIN max_offset_dist d
        ON g.cohort_bucket_id = d.cohort_bucket_id
        AND g.bucket_offset = d.bucket_offset
),
surv AS (
    SELECT
        cohort_bucket_id,
        bucket_offset,
        SUM(users_with_last_seen_here) OVER (
            PARTITION BY cohort_bucket_id
            ORDER BY bucket_offset DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS survival_users
    FROM base
)
SELECT
    s.cohort_bucket_id,
    bc.bucket_label AS cohort_label,
    s.bucket_offset,
    '+' || CAST(s.bucket_offset AS VARCHAR) AS offset_label,
    cs.cohort_users,
    s.survival_users,
    s.survival_users * 1.0 / NULLIF(cs.cohort_users, 0) AS survival_rate
FROM surv s
JOIN cohort_sizes cs
    ON s.cohort_bucket_id = cs.cohort_bucket_id
LEFT JOIN bucket_calendar bc
    ON s.cohort_bucket_id = bc.bucket_id
ORDER BY
    s.cohort_bucket_id,
    s.bucket_offset
""")

survival_long_df = con.execute("SELECT * FROM survival_long").df()
display(survival_long_df.head(30))

survival_long_df.to_csv(
    os.path.join(OUTPUT_DIR, "06_survival_return_after_long.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 9. WEIGHTED AVG RETENTION CURVES
# ============================================================

print("\n[6/8] Building average curves...")

avg_exact_df = con.execute("""
SELECT
    bucket_offset,
    '+' || CAST(bucket_offset AS VARCHAR) AS offset_label,
    SUM(active_users) AS active_users,
    SUM(cohort_users) AS eligible_cohort_users,
    SUM(active_users) * 1.0 / NULLIF(SUM(cohort_users), 0) AS weighted_retention_rate,
    AVG(retention_rate) AS unweighted_mean_retention_rate
FROM retention_long
GROUP BY bucket_offset
ORDER BY bucket_offset
""").df()

avg_survival_df = con.execute("""
SELECT
    bucket_offset,
    '+' || CAST(bucket_offset AS VARCHAR) AS offset_label,
    SUM(survival_users) AS survival_users,
    SUM(cohort_users) AS eligible_cohort_users,
    SUM(survival_users) * 1.0 / NULLIF(SUM(cohort_users), 0) AS weighted_survival_rate,
    AVG(survival_rate) AS unweighted_mean_survival_rate
FROM survival_long
GROUP BY bucket_offset
ORDER BY bucket_offset
""").df()

display(avg_exact_df)
display(avg_survival_df)

avg_exact_df.to_csv(
    os.path.join(OUTPUT_DIR, "07_avg_exact_retention_curve.csv"),
    index=False,
    encoding="utf-8-sig"
)

avg_survival_df.to_csv(
    os.path.join(OUTPUT_DIR, "08_avg_survival_return_after_curve.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 10. OPTIONAL: CITY / CATEGORY SWITCHING
# ============================================================

switch_avg_df = None
switch_long_df = None

if DO_SWITCH_ANALYSIS:
    print("\n[7/8] Building city/category switching analysis...")

    # Dominant city per user-bucket
    con.execute(f"""
    CREATE OR REPLACE TABLE user_bucket_city_pref AS
    WITH src AS (
        SELECT
            CAST(user_id AS VARCHAR) AS user_id,

            COALESCE(
                TRY_CAST(date AS DATE),
                CAST(event_ts AS DATE)
            ) AS event_date,

            CASE
                WHEN city_name IS NULL OR TRIM(CAST(city_name AS VARCHAR)) = ''
                THEN NULL
                ELSE TRIM(CAST(city_name AS VARCHAR))
            END AS city_name_clean

        FROM read_parquet('{EVENTS_GLOB}')
        WHERE user_id IS NOT NULL
            {login_filter_sql}
            {active_filter_sql}
    ),
    base AS (
        SELECT
            user_id,
            city_name_clean,

            CAST(
                EXTRACT(YEAR FROM event_date) * 24
                + (EXTRACT(MONTH FROM event_date) - 1) * 2
                + CASE
                    WHEN EXTRACT(DAY FROM event_date) <= 15 THEN 0
                    ELSE 1
                  END
                AS BIGINT
            ) AS bucket_id

        FROM src
        WHERE event_date IS NOT NULL
            AND city_name_clean IS NOT NULL
            {date_filter_sql}
    ),
    cnt AS (
        SELECT
            user_id,
            bucket_id,
            city_name_clean,
            COUNT(*) AS n_events
        FROM base
        GROUP BY user_id, bucket_id, city_name_clean
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY user_id, bucket_id
                ORDER BY n_events DESC, city_name_clean
            ) AS rn
        FROM cnt
    )
    SELECT
        user_id,
        bucket_id,
        city_name_clean AS city_mode,
        n_events AS city_mode_events
    FROM ranked
    WHERE rn = 1
    """)

    # Dominant category per user-bucket
    con.execute(f"""
    CREATE OR REPLACE TABLE user_bucket_category_pref AS
    WITH src AS (
        SELECT
            CAST(user_id AS VARCHAR) AS user_id,

            COALESCE(
                TRY_CAST(date AS DATE),
                CAST(event_ts AS DATE)
            ) AS event_date,

            TRY_CAST(category AS INTEGER) AS category_clean

        FROM read_parquet('{EVENTS_GLOB}')
        WHERE user_id IS NOT NULL
            {login_filter_sql}
            {active_filter_sql}
    ),
    base AS (
        SELECT
            user_id,
            category_clean,

            CAST(
                EXTRACT(YEAR FROM event_date) * 24
                + (EXTRACT(MONTH FROM event_date) - 1) * 2
                + CASE
                    WHEN EXTRACT(DAY FROM event_date) <= 15 THEN 0
                    ELSE 1
                  END
                AS BIGINT
            ) AS bucket_id

        FROM src
        WHERE event_date IS NOT NULL
            AND category_clean IS NOT NULL
            {date_filter_sql}
    ),
    cnt AS (
        SELECT
            user_id,
            bucket_id,
            category_clean,
            COUNT(*) AS n_events
        FROM base
        GROUP BY user_id, bucket_id, category_clean
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY user_id, bucket_id
                ORDER BY n_events DESC, category_clean
            ) AS rn
        FROM cnt
    )
    SELECT
        user_id,
        bucket_id,
        category_clean AS category_mode,
        n_events AS category_mode_events
    FROM ranked
    WHERE rn = 1
    """)

    con.execute("""
    CREATE OR REPLACE TABLE user_cohort_pref AS
    SELECT
        c.user_id,
        c.cohort_bucket_id,
        city.city_mode AS cohort_city_mode,
        cat.category_mode AS cohort_category_mode
    FROM user_cohort c
    LEFT JOIN user_bucket_city_pref city
        ON c.user_id = city.user_id
        AND c.cohort_bucket_id = city.bucket_id
    LEFT JOIN user_bucket_category_pref cat
        ON c.user_id = cat.user_id
        AND c.cohort_bucket_id = cat.bucket_id
    """)

    con.execute("""
    CREATE OR REPLACE TABLE switch_long AS
    SELECT
        c.cohort_bucket_id,
        bc.bucket_label AS cohort_label,

        uba.bucket_id - c.cohort_bucket_id AS bucket_offset,
        '+' || CAST(uba.bucket_id - c.cohort_bucket_id AS VARCHAR) AS offset_label,

        COUNT(*) AS active_user_buckets,

        SUM(
            CASE
                WHEN cp.cohort_city_mode IS NOT NULL
                     AND city.city_mode IS NOT NULL
                THEN 1 ELSE 0
            END
        ) AS city_comparable_users,

        SUM(
            CASE
                WHEN cp.cohort_city_mode IS NOT NULL
                     AND city.city_mode IS NOT NULL
                     AND city.city_mode <> cp.cohort_city_mode
                THEN 1 ELSE 0
            END
        ) AS city_switched_users,

        SUM(
            CASE
                WHEN cp.cohort_category_mode IS NOT NULL
                     AND cat.category_mode IS NOT NULL
                THEN 1 ELSE 0
            END
        ) AS category_comparable_users,

        SUM(
            CASE
                WHEN cp.cohort_category_mode IS NOT NULL
                     AND cat.category_mode IS NOT NULL
                     AND cat.category_mode <> cp.cohort_category_mode
                THEN 1 ELSE 0
            END
        ) AS category_switched_users

    FROM user_bucket_activity uba
    JOIN user_cohort c
        ON uba.user_id = c.user_id
    LEFT JOIN user_cohort_pref cp
        ON uba.user_id = cp.user_id
    LEFT JOIN user_bucket_city_pref city
        ON uba.user_id = city.user_id
        AND uba.bucket_id = city.bucket_id
    LEFT JOIN user_bucket_category_pref cat
        ON uba.user_id = cat.user_id
        AND uba.bucket_id = cat.bucket_id
    LEFT JOIN bucket_calendar bc
        ON c.cohort_bucket_id = bc.bucket_id
    WHERE uba.bucket_id >= c.cohort_bucket_id
    GROUP BY
        c.cohort_bucket_id,
        bc.bucket_label,
        uba.bucket_id - c.cohort_bucket_id
    ORDER BY
        c.cohort_bucket_id,
        bucket_offset
    """)

    switch_long_df = con.execute("""
    SELECT
        *,
        city_switched_users * 1.0 / NULLIF(city_comparable_users, 0) AS city_switch_rate,
        category_switched_users * 1.0 / NULLIF(category_comparable_users, 0) AS category_switch_rate
    FROM switch_long
    """).df()

    switch_avg_df = con.execute("""
    SELECT
        bucket_offset,
        '+' || CAST(bucket_offset AS VARCHAR) AS offset_label,

        SUM(city_switched_users) AS city_switched_users,
        SUM(city_comparable_users) AS city_comparable_users,
        SUM(city_switched_users) * 1.0 / NULLIF(SUM(city_comparable_users), 0) AS city_switch_rate,

        SUM(category_switched_users) AS category_switched_users,
        SUM(category_comparable_users) AS category_comparable_users,
        SUM(category_switched_users) * 1.0 / NULLIF(SUM(category_comparable_users), 0) AS category_switch_rate

    FROM switch_long
    GROUP BY bucket_offset
    ORDER BY bucket_offset
    """).df()

    display(switch_long_df.head(30))
    display(switch_avg_df)

    switch_long_df.to_csv(
        os.path.join(OUTPUT_DIR, "09_switch_city_category_long.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    switch_avg_df.to_csv(
        os.path.join(OUTPUT_DIR, "10_switch_city_category_avg_curve.csv"),
        index=False,
        encoding="utf-8-sig"
    )

# ============================================================
# 11. VISUALIZATION
# ============================================================

print("\n[8/8] Plotting charts...")

def save_plotly(fig, name):
    html_path = os.path.join(OUTPUT_DIR, f"{name}.html")
    fig.write_html(html_path)
    print("[SAVE HTML]", html_path)

    # PNG optional, nếu Kaggle thiếu kaleido thì bỏ qua
    try:
        png_path = os.path.join(OUTPUT_DIR, f"{name}.png")
        fig.write_image(png_path, scale=2)
        print("[SAVE PNG ]", png_path)
    except Exception as e:
        print(f"[SKIP PNG] {name}: {str(e)[:120]}")

# ----------------------------
# 11.1 Retention heatmap
# ----------------------------

heatmap_df = retention_matrix_df.copy()

# Text chỉ hiển thị % nếu số cohort/offset không quá dày
show_text = (heatmap_df.shape[0] <= 30 and heatmap_df.shape[1] <= 35)

fig_heatmap = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="Blues",
    zmin=0,
    zmax=1,
    text_auto=".1%" if show_text else False,
    title=f"Half-month User Retention Heatmap — LOGIN_MODE={LOGIN_MODE}"
)

fig_heatmap.update_layout(
    xaxis_title="Bucket offset from first seen cohort",
    yaxis_title="User cohort first seen bucket",
    height=max(500, 26 * len(heatmap_df.index)),
    width=max(900, 55 * len(heatmap_df.columns)),
)

fig_heatmap.show()
save_plotly(fig_heatmap, "01_retention_heatmap_exact")

# ----------------------------
# 11.2 Average exact retention curve
# ----------------------------

fig_avg_exact = px.line(
    avg_exact_df,
    x="offset_label",
    y="weighted_retention_rate",
    markers=True,
    title="Weighted Average Exact Retention by Half-month Offset",
    hover_data={
        "active_users": ":,",
        "eligible_cohort_users": ":,",
        "weighted_retention_rate": ":.2%",
        "unweighted_mean_retention_rate": ":.2%",
    }
)

fig_avg_exact.update_layout(
    xaxis_title="Bucket offset",
    yaxis_title="Weighted exact retention rate",
    yaxis_tickformat=".0%",
    height=500,
    width=1000
)

fig_avg_exact.show()
save_plotly(fig_avg_exact, "02_avg_exact_retention_curve")

# ----------------------------
# 11.3 Survival / return-after curve
# ----------------------------

fig_avg_survival = px.line(
    avg_survival_df,
    x="offset_label",
    y="weighted_survival_rate",
    markers=True,
    title="Weighted Survival / Return-after Rate by Half-month Offset",
    hover_data={
        "survival_users": ":,",
        "eligible_cohort_users": ":,",
        "weighted_survival_rate": ":.2%",
        "unweighted_mean_survival_rate": ":.2%",
    }
)

fig_avg_survival.update_layout(
    xaxis_title="Bucket offset",
    yaxis_title="Return-after / survival rate",
    yaxis_tickformat=".0%",
    height=500,
    width=1000
)

fig_avg_survival.show()
save_plotly(fig_avg_survival, "03_avg_survival_return_after_curve")

# ----------------------------
# 11.4 Cohort size
# ----------------------------

fig_cohort = px.bar(
    cohort_sizes_df,
    x="cohort_label",
    y="cohort_users",
    title="New User Cohort Size by Half-month Bucket",
    hover_data={
        "cohort_start": True,
        "cohort_end": True,
        "cohort_users": ":,",
    }
)

fig_cohort.update_layout(
    xaxis_title="First seen bucket",
    yaxis_title="New users",
    height=500,
    width=1000
)

fig_cohort.show()
save_plotly(fig_cohort, "04_cohort_size_by_bucket")

# ----------------------------
# 11.5 Active users by bucket
# ----------------------------

active_by_bucket_df = con.execute("""
SELECT
    bc.bucket_label,
    bc.bucket_start,
    bc.bucket_end,
    COUNT(DISTINCT uba.user_id) AS active_users,
    SUM(uba.events) AS events,
    SUM(uba.positive_events) AS positive_events,
    SUM(uba.strict_contact_events) AS strict_contact_events
FROM user_bucket_activity uba
LEFT JOIN bucket_calendar bc
    ON uba.bucket_id = bc.bucket_id
GROUP BY
    bc.bucket_label,
    bc.bucket_start,
    bc.bucket_end
ORDER BY
    bc.bucket_start
""").df()

active_by_bucket_df.to_csv(
    os.path.join(OUTPUT_DIR, "11_active_users_by_bucket.csv"),
    index=False,
    encoding="utf-8-sig"
)

fig_active = px.line(
    active_by_bucket_df,
    x="bucket_label",
    y="active_users",
    markers=True,
    title="Active Users by Half-month Bucket",
    hover_data={
        "bucket_start": True,
        "bucket_end": True,
        "active_users": ":,",
        "events": ":,",
        "positive_events": ":,",
        "strict_contact_events": ":,",
    }
)

fig_active.update_layout(
    xaxis_title="Half-month bucket",
    yaxis_title="Active users",
    height=500,
    width=1000
)

fig_active.show()
save_plotly(fig_active, "05_active_users_by_bucket")

# ----------------------------
# 11.6 Switching curve
# ----------------------------

if DO_SWITCH_ANALYSIS and switch_avg_df is not None:
    switch_plot_df = switch_avg_df.melt(
        id_vars=["bucket_offset", "offset_label"],
        value_vars=["city_switch_rate", "category_switch_rate"],
        var_name="switch_type",
        value_name="switch_rate"
    )

    switch_plot_df["switch_type"] = switch_plot_df["switch_type"].map({
        "city_switch_rate": "Dominant city changed",
        "category_switch_rate": "Dominant category changed"
    })

    fig_switch = px.line(
        switch_plot_df,
        x="offset_label",
        y="switch_rate",
        color="switch_type",
        markers=True,
        title="Among Retained Users: City / Category Switching vs First-seen Bucket",
        hover_data={
            "bucket_offset": True,
            "switch_rate": ":.2%",
        }
    )

    fig_switch.update_layout(
        xaxis_title="Bucket offset",
        yaxis_title="Switch rate among comparable active users",
        yaxis_tickformat=".0%",
        height=500,
        width=1000
    )

    fig_switch.show()
    save_plotly(fig_switch, "06_city_category_switching_curve")

# ============================================================
# 12. PRINT INTERPRETATION GUIDE
# ============================================================

print("\nDONE.")
print("Output folder:", OUTPUT_DIR)

print("""
CÁCH ĐỌC NHANH:

1) 01_retention_heatmap_exact.html
   - Mỗi row = nhóm user xuất hiện lần đầu ở một bucket nửa tháng.
   - +0 = chính bucket đầu tiên, luôn gần 100%.
   - +1 = sau nửa tháng kế tiếp còn active bao nhiêu %.
   - +2 = sau 1 tháng còn active bao nhiêu %.
   - +4 = sau 2 tháng còn active bao nhiêu %.

2) 02_avg_exact_retention_curve.html
   - Retention đúng tại từng offset.
   - Nếu +1 rớt mạnh: user vào xem một lần rồi biến mất nhanh.
   - Nếu +2/+3 còn cao: user có chu kỳ tìm BĐS dài hơn.

3) 03_avg_survival_return_after_curve.html
   - User có quay lại ít nhất một lần từ offset đó trở đi không.
   - Metric này hợp với BĐS hơn exact retention, vì người tìm nhà không nhất thiết tuần nào/nửa tháng nào cũng vào.

4) 04_cohort_size_by_bucket.html
   - Bucket nào kéo được nhiều user mới.
   - Nếu cohort lớn nhưng retention thấp: acquisition tốt nhưng giữ chân yếu.

5) 05_active_users_by_bucket.html
   - Tổng active users theo timeline nửa tháng.
   - Dùng để nhìn seasonality / trend tăng giảm tổng thể.

6) 06_city_category_switching_curve.html
   - Trong số user còn quay lại, bao nhiêu % đổi city/category chính so với bucket đầu tiên.
   - Nếu city_switch_rate tăng theo offset: user ban đầu search một khu, sau đó mở rộng/đổi địa bàn.
   - Nếu category_switch_rate tăng: user đang đổi nhu cầu, ví dụ từ thuê phòng sang chung cư/nhà ở.
""")

print("Important CSV files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".csv"):
        print("-", f)

con.close()

EVENTS_PATH: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events
EVENTS_GLOB: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events/*.parquet
OUTPUT_DIR : /kaggle/working/user_retention_half_month


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,total_rows,login_rows,non_login_rows
0,2025-11-09,2026-04-09,161731336,102098089.0,59633247.0



[1/8] Building user_bucket_activity...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

[SAVE HTML] /kaggle/working/user_retention_half_month/02_avg_exact_retention_curve.html
[SKIP PNG] 02_avg_exact_retention_curve: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip insta


[SAVE HTML] /kaggle/working/user_retention_half_month/03_avg_survival_return_after_curve.html
[SKIP PNG] 03_avg_survival_return_after_curve: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip insta


[SAVE HTML] /kaggle/working/user_retention_half_month/04_cohort_size_by_bucket.html
[SKIP PNG] 04_cohort_size_by_bucket: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip insta


[SAVE HTML] /kaggle/working/user_retention_half_month/05_active_users_by_bucket.html
[SKIP PNG] 05_active_users_by_bucket: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip insta


[SAVE HTML] /kaggle/working/user_retention_half_month/06_city_category_switching_curve.html
[SKIP PNG] 06_city_category_switching_curve: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip insta

DONE.
Output folder: /kaggle/working/user_retention_half_month

CÁCH ĐỌC NHANH:

1) 01_retention_heatmap_exact.html
   - Mỗi row = nhóm user xuất hiện lần đầu ở một bucket nửa tháng.
   - +0 = chính bucket đầu tiên, luôn gần 100%.
   - +1 = sau nửa tháng kế tiếp còn active bao nhiêu %.
   - +2 = sau 1 tháng còn active bao nhiêu %.
   - +4 = sau 2 tháng còn active bao nhiêu %.

2) 02_avg_exact_retention_curve.html
   - Retention đúng tại từng offset.
   - Nếu +1 rớt mạnh: user vào xem một lần rồi biến mất nhanh.
   - Nếu +2/+3 còn cao: user có chu kỳ tìm BĐS dài hơn.

3) 03_avg_survival_return_after_curve.html
   - User có quay lại ít nhất một lần từ offset đó trở đi không.
   - Metric này hợp với BĐS hơn exact retention, vì người t

# **check thêm thuê với mua**, check coi sell hay let & check có coi nhiều city hông (nhiều city -> cò) (đã có chuyển city & category)

**user & cò**

In [1]:
# ============================================================
# DATATHON 2026 — LOW-DISK RETENTION DEEP DIVE
# sell vs let + multi-city behavior
# ============================================================
# Fix lỗi:
# IOException: No space left on device / duckdb_tmp
#
# Idea:
# - Không tạo bảng combo lớn user x bucket x city x category x ad_type x seller_type
# - Scan từng parquet file
# - Aggregate nhỏ theo từng chiều riêng:
#     user_bucket overall
#     user_bucket_city
#     user_bucket_category
#     user_bucket_ad_type
# - Merge lại sau
#
# Output:
# - Switch city/category/sell-let
# - Retention by first-seen let/sell
# - City switch vs multi-city diagnosis
# ============================================================

import os
import sys
import glob
import shutil
import subprocess
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 0. CONFIG
# ============================================================

OUTPUT_DIR = "/kaggle/working/retention_lowdisk_sell_let_multicity"

# Xóa output cũ để dọn duckdb_tmp đã làm đầy disk
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

PARTS_DIR = os.path.join(OUTPUT_DIR, "parts")
os.makedirs(PARTS_DIR, exist_ok=True)

# Dataset của bạn đang tách:
# Datathon2026_1: dim_listing
# Datathon2026_2: fact_user_events
SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/input/Datathon2026_1",
    "/kaggle/input/Datathon2026_2",
    "/kaggle/input/datathon2026-1",
    "/kaggle/input/datathon2026-2",
    "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1",
    "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2",
    "/kaggle/input/datasets/luquang231/datathonvinu",
]

EVENTS_DIR_NAMES = ["fact_user_events"]
DIM_DIR_NAMES = ["dim_listing_raw_clean", "dim_listing_clean", "dim_listing"]

LOGIN_MODE = "login"       # "login" | "non-login" | "all"

START_DATE = None
END_DATE = None

# Nếu muốn train window:
# START_DATE = "2025-11-09"
# END_DATE = "2026-04-09"

ACTIVE_EVENT_TYPES = None
# ACTIVE_EVENT_TYPES = ["pageview"]
# ACTIVE_EVENT_TYPES = ["view_phone", "contact_chat", "contact_zalo", "contact_sms"]

POSITIVE_EVENT_TYPES = [
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms"
]

STRICT_CONTACT_EVENT_TYPES = [
    "view_phone", "contact_chat", "contact_zalo", "contact_sms"
]

DUCKDB_MEMORY_LIMIT = "4GB"
DUCKDB_THREADS = 1

# Giới hạn temp thấp để nó không ăn hết disk
DUCKDB_MAX_TEMP_DIR_SIZE = "6GB"

# Nếu chỉ test nhanh, set số nhỏ, ví dụ 20.
# Chạy full thì để None.
MAX_EVENT_FILES = None

# ============================================================
# 1. SETUP
# ============================================================

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

try:
    import plotly.express as px
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly"])
    import plotly.express as px


def find_existing_subdir_anywhere(target_names, search_roots):
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for name in target_names:
            p = os.path.join(root, name)
            if os.path.exists(p):
                return p

    for root in search_roots:
        if not os.path.exists(root):
            continue
        for base, dirs, files in os.walk(root):
            for name in target_names:
                if name in dirs:
                    return os.path.join(base, name)

    raise FileNotFoundError(f"Không tìm thấy folder nào trong list: {target_names}")


def sql_str_list(values):
    return ", ".join(["'" + str(v).replace("'", "''").lower() + "'" for v in values])


EVENTS_PATH = find_existing_subdir_anywhere(EVENTS_DIR_NAMES, SEARCH_ROOTS)
DIM_PATH = find_existing_subdir_anywhere(DIM_DIR_NAMES, SEARCH_ROOTS)

EVENT_FILES = sorted(glob.glob(os.path.join(EVENTS_PATH, "*.parquet")))
DIM_GLOB = os.path.join(DIM_PATH, "*.parquet").replace("\\", "/")

if MAX_EVENT_FILES is not None:
    EVENT_FILES = EVENT_FILES[:MAX_EVENT_FILES]

print("EVENTS_PATH:", EVENTS_PATH)
print("DIM_PATH   :", DIM_PATH)
print("EVENT_FILES:", len(EVENT_FILES))
print("OUTPUT_DIR :", OUTPUT_DIR)

duckdb_path = os.path.join(OUTPUT_DIR, "retention_lowdisk.duckdb")
tmp_dir = os.path.join(OUTPUT_DIR, "duckdb_tmp")
os.makedirs(tmp_dir, exist_ok=True)

con = duckdb.connect(duckdb_path)
con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA temp_directory='{tmp_dir}'")
con.execute(f"PRAGMA max_temp_directory_size='{DUCKDB_MAX_TEMP_DIR_SIZE}'")
con.execute("SET preserve_insertion_order=false")

login_filter_sql = ""
if LOGIN_MODE == "login":
    login_filter_sql = "AND LOWER(TRIM(CAST(e.is_login AS VARCHAR))) = 'login'"
elif LOGIN_MODE == "non-login":
    login_filter_sql = "AND LOWER(TRIM(CAST(e.is_login AS VARCHAR))) = 'non-login'"
elif LOGIN_MODE == "all":
    login_filter_sql = ""
else:
    raise ValueError("LOGIN_MODE phải là login / non-login / all")

active_filter_sql = ""
if ACTIVE_EVENT_TYPES is not None:
    active_filter_sql = f"""
    AND LOWER(TRIM(CAST(e.event_type AS VARCHAR))) IN ({sql_str_list(ACTIVE_EVENT_TYPES)})
    """

date_filter_sql = ""
if START_DATE is not None:
    date_filter_sql += f" AND event_date >= DATE '{START_DATE}' "
if END_DATE is not None:
    date_filter_sql += f" AND event_date <= DATE '{END_DATE}' "

positive_sql = sql_str_list(POSITIVE_EVENT_TYPES)
strict_sql = sql_str_list(STRICT_CONTACT_EVENT_TYPES)

# ============================================================
# 2. BUILD DIM LITE
# ============================================================

print("\n[1/8] Build dim_lite...")

con.execute(f"""
CREATE OR REPLACE TABLE dim_lite AS
SELECT
    CAST(item_id AS VARCHAR) AS item_id,

    TRY_CAST(category AS INTEGER) AS dim_category,

    CASE
        WHEN city_name IS NULL OR TRIM(CAST(city_name AS VARCHAR)) = ''
        THEN NULL
        ELSE TRIM(CAST(city_name AS VARCHAR))
    END AS dim_city_name,

    CASE
        WHEN LOWER(TRIM(CAST(ad_type AS VARCHAR))) IN ('sell', 'let')
        THEN LOWER(TRIM(CAST(ad_type AS VARCHAR)))
        ELSE NULL
    END AS ad_type_clean

FROM read_parquet('{DIM_GLOB}')
WHERE item_id IS NOT NULL
""")

dim_df = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT item_id) AS items,
    SUM(CASE WHEN ad_type_clean = 'sell' THEN 1 ELSE 0 END) AS sell_items,
    SUM(CASE WHEN ad_type_clean = 'let' THEN 1 ELSE 0 END) AS let_items,
    SUM(CASE WHEN ad_type_clean IS NULL THEN 1 ELSE 0 END) AS unknown_ad_type_items
FROM dim_lite
""").df()

display(dim_df)
dim_df.to_csv(os.path.join(OUTPUT_DIR, "00_dim_check.csv"), index=False, encoding="utf-8-sig")

# ============================================================
# 3. PER-FILE AGGREGATION
# ============================================================

print("\n[2/8] Per-file aggregation...")

def qpath(p):
    return p.replace("\\", "/").replace("'", "''")


for i, fp in enumerate(EVENT_FILES):
    fp_sql = qpath(fp)

    ub_out = qpath(os.path.join(PARTS_DIR, f"ub_{i:04d}.parquet"))
    city_out = qpath(os.path.join(PARTS_DIR, f"city_{i:04d}.parquet"))
    cat_out = qpath(os.path.join(PARTS_DIR, f"cat_{i:04d}.parquet"))
    ad_out = qpath(os.path.join(PARTS_DIR, f"ad_{i:04d}.parquet"))

    print(f"[{i+1}/{len(EVENT_FILES)}] {os.path.basename(fp)}")

    base_cte = f"""
    WITH src AS (
        SELECT
            CAST(e.user_id AS VARCHAR) AS user_id,

            COALESCE(
                TRY_CAST(e.date AS DATE),
                CAST(e.event_ts AS DATE)
            ) AS event_date,

            LOWER(TRIM(CAST(e.event_type AS VARCHAR))) AS event_type_clean,

            COALESCE(
                CASE
                    WHEN e.city_name IS NULL OR TRIM(CAST(e.city_name AS VARCHAR)) = ''
                    THEN NULL
                    ELSE TRIM(CAST(e.city_name AS VARCHAR))
                END,
                d.dim_city_name
            ) AS city_clean,

            COALESCE(
                TRY_CAST(e.category AS INTEGER),
                d.dim_category
            ) AS category_clean,

            d.ad_type_clean AS ad_type_clean

        FROM read_parquet('{fp_sql}') e
        LEFT JOIN dim_lite d
            ON CAST(e.item_id AS VARCHAR) = d.item_id
        WHERE e.user_id IS NOT NULL
            {login_filter_sql}
            {active_filter_sql}
    ),
    base AS (
        SELECT
            user_id,
            event_type_clean,
            city_clean,
            category_clean,
            ad_type_clean,

            CAST(
                EXTRACT(YEAR FROM event_date) * 24
                + (EXTRACT(MONTH FROM event_date) - 1) * 2
                + CASE
                    WHEN EXTRACT(DAY FROM event_date) <= 15 THEN 0
                    ELSE 1
                  END
                AS BIGINT
            ) AS bucket_id

        FROM src
        WHERE event_date IS NOT NULL
            {date_filter_sql}
    )
    """

    # 3.1 user-bucket overall
    con.execute(f"""
    COPY (
        {base_cte}
        SELECT
            user_id,
            bucket_id,
            COUNT(*) AS events,
            SUM(CASE WHEN event_type_clean IN ({positive_sql}) THEN 1 ELSE 0 END) AS positive_events,
            SUM(CASE WHEN event_type_clean IN ({strict_sql}) THEN 1 ELSE 0 END) AS strict_contact_events
        FROM base
        GROUP BY user_id, bucket_id
    )
    TO '{ub_out}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    # 3.2 city counts
    con.execute(f"""
    COPY (
        {base_cte}
        SELECT
            user_id,
            bucket_id,
            city_clean,
            COUNT(*) AS events,
            SUM(CASE WHEN event_type_clean IN ({positive_sql}) THEN 1 ELSE 0 END) AS positive_events
        FROM base
        WHERE city_clean IS NOT NULL
        GROUP BY user_id, bucket_id, city_clean
    )
    TO '{city_out}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    # 3.3 category counts
    con.execute(f"""
    COPY (
        {base_cte}
        SELECT
            user_id,
            bucket_id,
            category_clean,
            COUNT(*) AS events,
            SUM(CASE WHEN event_type_clean IN ({positive_sql}) THEN 1 ELSE 0 END) AS positive_events
        FROM base
        WHERE category_clean IS NOT NULL
        GROUP BY user_id, bucket_id, category_clean
    )
    TO '{cat_out}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    # 3.4 ad_type counts: sell / let
    con.execute(f"""
    COPY (
        {base_cte}
        SELECT
            user_id,
            bucket_id,
            ad_type_clean,
            COUNT(*) AS events,
            SUM(CASE WHEN event_type_clean IN ({positive_sql}) THEN 1 ELSE 0 END) AS positive_events
        FROM base
        WHERE ad_type_clean IN ('sell', 'let')
        GROUP BY user_id, bucket_id, ad_type_clean
    )
    TO '{ad_out}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

print("\nPer-file aggregation done.")

# ============================================================
# 4. MERGE PARTS INTO FINAL PROFILE
# ============================================================

print("\n[3/8] Merge user-bucket overall...")

UB_GLOB = qpath(os.path.join(PARTS_DIR, "ub_*.parquet"))
CITY_GLOB = qpath(os.path.join(PARTS_DIR, "city_*.parquet"))
CAT_GLOB = qpath(os.path.join(PARTS_DIR, "cat_*.parquet"))
AD_GLOB = qpath(os.path.join(PARTS_DIR, "ad_*.parquet"))

con.execute(f"""
CREATE OR REPLACE TABLE user_bucket_base AS
SELECT
    user_id,
    bucket_id,
    SUM(events) AS events,
    SUM(positive_events) AS positive_events,
    SUM(strict_contact_events) AS strict_contact_events
FROM read_parquet('{UB_GLOB}')
GROUP BY user_id, bucket_id
""")

# Xóa ub parts sau khi merge để giải phóng disk
for f in glob.glob(os.path.join(PARTS_DIR, "ub_*.parquet")):
    os.remove(f)

print("\n[4/8] Merge city profile...")

con.execute(f"""
CREATE OR REPLACE TABLE city_counts AS
SELECT
    user_id,
    bucket_id,
    city_clean,
    SUM(events) AS events,
    SUM(positive_events) AS positive_events
FROM read_parquet('{CITY_GLOB}')
GROUP BY user_id, bucket_id, city_clean
""")

con.execute("""
CREATE OR REPLACE TABLE city_profile AS
SELECT
    user_id,
    bucket_id,
    COUNT(*) AS distinct_cities_seen
FROM city_counts
GROUP BY user_id, bucket_id
""")

con.execute("""
CREATE OR REPLACE TABLE city_pref AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, bucket_id
            ORDER BY events DESC, positive_events DESC, city_clean
        ) AS rn
    FROM city_counts
)
SELECT
    user_id,
    bucket_id,
    city_clean AS city_mode,
    events AS city_mode_events
FROM ranked
WHERE rn = 1
""")

con.execute("DROP TABLE city_counts")
con.execute("CHECKPOINT")

for f in glob.glob(os.path.join(PARTS_DIR, "city_*.parquet")):
    os.remove(f)

print("\n[5/8] Merge category profile...")

con.execute(f"""
CREATE OR REPLACE TABLE cat_counts AS
SELECT
    user_id,
    bucket_id,
    category_clean,
    SUM(events) AS events,
    SUM(positive_events) AS positive_events
FROM read_parquet('{CAT_GLOB}')
GROUP BY user_id, bucket_id, category_clean
""")

con.execute("""
CREATE OR REPLACE TABLE cat_profile AS
SELECT
    user_id,
    bucket_id,
    COUNT(*) AS distinct_categories_seen
FROM cat_counts
GROUP BY user_id, bucket_id
""")

con.execute("""
CREATE OR REPLACE TABLE cat_pref AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, bucket_id
            ORDER BY events DESC, positive_events DESC, category_clean
        ) AS rn
    FROM cat_counts
)
SELECT
    user_id,
    bucket_id,
    category_clean AS category_mode,
    events AS category_mode_events
FROM ranked
WHERE rn = 1
""")

con.execute("DROP TABLE cat_counts")
con.execute("CHECKPOINT")

for f in glob.glob(os.path.join(PARTS_DIR, "cat_*.parquet")):
    os.remove(f)

print("\n[6/8] Merge ad_type profile...")

con.execute(f"""
CREATE OR REPLACE TABLE ad_counts AS
SELECT
    user_id,
    bucket_id,
    ad_type_clean,
    SUM(events) AS events,
    SUM(positive_events) AS positive_events
FROM read_parquet('{AD_GLOB}')
GROUP BY user_id, bucket_id, ad_type_clean
""")

con.execute("""
CREATE OR REPLACE TABLE ad_profile AS
SELECT
    user_id,
    bucket_id,
    COUNT(*) AS distinct_ad_types_seen
FROM ad_counts
GROUP BY user_id, bucket_id
""")

con.execute("""
CREATE OR REPLACE TABLE ad_pref AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, bucket_id
            ORDER BY events DESC, positive_events DESC, ad_type_clean
        ) AS rn
    FROM ad_counts
)
SELECT
    user_id,
    bucket_id,
    ad_type_clean AS ad_type_mode,
    events AS ad_type_mode_events
FROM ranked
WHERE rn = 1
""")

con.execute("DROP TABLE ad_counts")
con.execute("CHECKPOINT")

for f in glob.glob(os.path.join(PARTS_DIR, "ad_*.parquet")):
    os.remove(f)

# Remove empty parts dir
try:
    os.rmdir(PARTS_DIR)
except Exception:
    pass

# Final profile
con.execute("""
CREATE OR REPLACE TABLE user_bucket_profile AS
SELECT
    b.user_id,
    b.bucket_id,

    b.events,
    b.positive_events,
    b.strict_contact_events,

    COALESCE(cp.distinct_cities_seen, 0) AS distinct_cities_seen,
    COALESCE(kp.distinct_categories_seen, 0) AS distinct_categories_seen,
    COALESCE(ap.distinct_ad_types_seen, 0) AS distinct_ad_types_seen,

    city.city_mode,
    cat.category_mode,
    ad.ad_type_mode,

    CASE WHEN COALESCE(cp.distinct_cities_seen, 0) >= 2 THEN 1 ELSE 0 END AS is_multi_city_2plus,
    CASE WHEN COALESCE(cp.distinct_cities_seen, 0) >= 3 THEN 1 ELSE 0 END AS is_multi_city_3plus,
    CASE WHEN COALESCE(ap.distinct_ad_types_seen, 0) >= 2 THEN 1 ELSE 0 END AS is_mixed_sell_let

FROM user_bucket_base b
LEFT JOIN city_profile cp
    ON b.user_id = cp.user_id
    AND b.bucket_id = cp.bucket_id
LEFT JOIN cat_profile kp
    ON b.user_id = kp.user_id
    AND b.bucket_id = kp.bucket_id
LEFT JOIN ad_profile ap
    ON b.user_id = ap.user_id
    AND b.bucket_id = ap.bucket_id
LEFT JOIN city_pref city
    ON b.user_id = city.user_id
    AND b.bucket_id = city.bucket_id
LEFT JOIN cat_pref cat
    ON b.user_id = cat.user_id
    AND b.bucket_id = cat.bucket_id
LEFT JOIN ad_pref ad
    ON b.user_id = ad.user_id
    AND b.bucket_id = ad.bucket_id
""")

profile_check = con.execute("""
SELECT
    COUNT(*) AS user_bucket_rows,
    COUNT(DISTINCT user_id) AS users,
    MIN(bucket_id) AS min_bucket_id,
    MAX(bucket_id) AS max_bucket_id,
    AVG(distinct_cities_seen) AS avg_cities_per_user_bucket,
    AVG(distinct_categories_seen) AS avg_categories_per_user_bucket,
    AVG(distinct_ad_types_seen) AS avg_ad_types_per_user_bucket,
    SUM(is_multi_city_2plus) * 1.0 / COUNT(*) AS multi_city_2plus_rate,
    SUM(is_multi_city_3plus) * 1.0 / COUNT(*) AS multi_city_3plus_rate,
    SUM(is_mixed_sell_let) * 1.0 / COUNT(*) AS mixed_sell_let_rate
FROM user_bucket_profile
""").df()

display(profile_check)
profile_check.to_csv(os.path.join(OUTPUT_DIR, "01_user_bucket_profile_check.csv"), index=False, encoding="utf-8-sig")

# ============================================================
# 5. BUCKET CALENDAR + COHORT
# ============================================================

print("\n[7/8] Build cohort and retained state...")

con.execute("""
CREATE OR REPLACE TABLE bucket_calendar AS
WITH lim AS (
    SELECT MIN(bucket_id) AS min_bucket_id, MAX(bucket_id) AS max_bucket_id
    FROM user_bucket_profile
),
ids AS (
    SELECT r.bucket_id
    FROM lim
    CROSS JOIN LATERAL range(
        CAST(min_bucket_id AS BIGINT),
        CAST(max_bucket_id + 1 AS BIGINT)
    ) AS r(bucket_id)
),
parts AS (
    SELECT
        bucket_id,
        CAST(FLOOR(bucket_id / 24) AS INTEGER) AS yy,
        CAST(FLOOR((bucket_id % 24) / 2) + 1 AS INTEGER) AS mm,
        CAST(bucket_id % 2 AS INTEGER) AS half_idx
    FROM ids
),
starts AS (
    SELECT
        bucket_id,
        yy,
        mm,
        half_idx,
        CAST(
            CAST(yy AS VARCHAR)
            || '-'
            || LPAD(CAST(mm AS VARCHAR), 2, '0')
            || '-01'
            AS DATE
        )
        + CASE WHEN half_idx = 1 THEN INTERVAL '15 days' ELSE INTERVAL '0 days' END AS bucket_start
    FROM parts
)
SELECT
    bucket_id,
    CAST(bucket_start AS DATE) AS bucket_start,
    CASE
        WHEN half_idx = 0 THEN CAST(bucket_start + INTERVAL '14 days' AS DATE)
        ELSE CAST(DATE_TRUNC('month', bucket_start) + INTERVAL '1 month' - INTERVAL '1 day' AS DATE)
    END AS bucket_end,
    STRFTIME(bucket_start, '%Y-%m')
        || CASE WHEN half_idx = 0 THEN '-H1' ELSE '-H2' END AS bucket_label
FROM starts
ORDER BY bucket_id
""")

bucket_calendar_df = con.execute("SELECT * FROM bucket_calendar ORDER BY bucket_id").df()
display(bucket_calendar_df)
bucket_calendar_df.to_csv(os.path.join(OUTPUT_DIR, "02_bucket_calendar.csv"), index=False, encoding="utf-8-sig")

con.execute("""
CREATE OR REPLACE TABLE user_cohort AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY bucket_id
        ) AS rn
    FROM user_bucket_profile
)
SELECT
    user_id,
    bucket_id AS cohort_bucket_id,

    city_mode AS cohort_city_mode,
    category_mode AS cohort_category_mode,
    ad_type_mode AS cohort_ad_type_mode,

    distinct_cities_seen AS cohort_distinct_cities_seen,
    distinct_categories_seen AS cohort_distinct_categories_seen,
    distinct_ad_types_seen AS cohort_distinct_ad_types_seen,

    is_multi_city_2plus AS cohort_is_multi_city_2plus,
    is_multi_city_3plus AS cohort_is_multi_city_3plus,
    is_mixed_sell_let AS cohort_is_mixed_sell_let
FROM ranked
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TABLE retained_user_state AS
SELECT
    c.user_id,

    c.cohort_bucket_id,
    bc0.bucket_label AS cohort_label,

    p.bucket_id AS active_bucket_id,
    bc1.bucket_label AS active_bucket_label,

    p.bucket_id - c.cohort_bucket_id AS bucket_offset,
    '+' || CAST(p.bucket_id - c.cohort_bucket_id AS VARCHAR) AS offset_label,

    c.cohort_city_mode,
    c.cohort_category_mode,
    c.cohort_ad_type_mode,

    c.cohort_distinct_cities_seen,
    c.cohort_distinct_categories_seen,
    c.cohort_distinct_ad_types_seen,

    c.cohort_is_multi_city_2plus,
    c.cohort_is_multi_city_3plus,
    c.cohort_is_mixed_sell_let,

    p.city_mode,
    p.category_mode,
    p.ad_type_mode,

    p.events,
    p.positive_events,
    p.strict_contact_events,

    p.distinct_cities_seen,
    p.distinct_categories_seen,
    p.distinct_ad_types_seen,

    p.is_multi_city_2plus,
    p.is_multi_city_3plus,
    p.is_mixed_sell_let,

    CASE
        WHEN c.cohort_city_mode IS NOT NULL
             AND p.city_mode IS NOT NULL
             AND c.cohort_city_mode <> p.city_mode
        THEN 1 ELSE 0
    END AS city_switched,

    CASE
        WHEN c.cohort_category_mode IS NOT NULL
             AND p.category_mode IS NOT NULL
             AND c.cohort_category_mode <> p.category_mode
        THEN 1 ELSE 0
    END AS category_switched,

    CASE
        WHEN c.cohort_ad_type_mode IN ('sell', 'let')
             AND p.ad_type_mode IN ('sell', 'let')
             AND c.cohort_ad_type_mode <> p.ad_type_mode
        THEN 1 ELSE 0
    END AS ad_type_switched

FROM user_cohort c
JOIN user_bucket_profile p
    ON c.user_id = p.user_id
    AND p.bucket_id >= c.cohort_bucket_id
LEFT JOIN bucket_calendar bc0
    ON c.cohort_bucket_id = bc0.bucket_id
LEFT JOIN bucket_calendar bc1
    ON p.bucket_id = bc1.bucket_id
""")

state_check = con.execute("""
SELECT
    COUNT(*) AS retained_user_bucket_rows,
    COUNT(DISTINCT user_id) AS users,
    MIN(bucket_offset) AS min_offset,
    MAX(bucket_offset) AS max_offset
FROM retained_user_state
""").df()

display(state_check)
state_check.to_csv(os.path.join(OUTPUT_DIR, "03_retained_state_check.csv"), index=False, encoding="utf-8-sig")

# ============================================================
# 6. SUMMARY TABLES
# ============================================================

print("\n[8/8] Build summaries...")

overall_offset_df = con.execute("""
SELECT
    bucket_offset,
    offset_label,

    COUNT(*) AS active_user_buckets,

    SUM(city_switched) AS city_switched_users,
    SUM(category_switched) AS category_switched_users,
    SUM(ad_type_switched) AS ad_type_switched_users,

    SUM(is_multi_city_2plus) AS current_multi_city_2plus_users,
    SUM(is_multi_city_3plus) AS current_multi_city_3plus_users,
    SUM(is_mixed_sell_let) AS current_mixed_sell_let_users,

    SUM(CASE WHEN city_switched = 1 AND is_multi_city_2plus = 1 THEN 1 ELSE 0 END)
        AS city_switch_and_multi_city_2plus,

    SUM(CASE WHEN city_switched = 1 AND is_multi_city_3plus = 1 THEN 1 ELSE 0 END)
        AS city_switch_and_multi_city_3plus,

    SUM(CASE WHEN category_switched = 1 AND ad_type_switched = 1 THEN 1 ELSE 0 END)
        AS category_switch_and_ad_type_switch,

    SUM(CASE WHEN category_switched = 1 AND is_mixed_sell_let = 1 THEN 1 ELSE 0 END)
        AS category_switch_and_mixed_sell_let

FROM retained_user_state
GROUP BY bucket_offset, offset_label
ORDER BY bucket_offset
""").df()

overall_offset_df["city_switch_rate"] = (
    overall_offset_df["city_switched_users"] /
    overall_offset_df["active_user_buckets"].replace(0, np.nan)
)
overall_offset_df["category_switch_rate"] = (
    overall_offset_df["category_switched_users"] /
    overall_offset_df["active_user_buckets"].replace(0, np.nan)
)
overall_offset_df["ad_type_switch_rate"] = (
    overall_offset_df["ad_type_switched_users"] /
    overall_offset_df["active_user_buckets"].replace(0, np.nan)
)
overall_offset_df["current_multi_city_2plus_rate"] = (
    overall_offset_df["current_multi_city_2plus_users"] /
    overall_offset_df["active_user_buckets"].replace(0, np.nan)
)
overall_offset_df["current_multi_city_3plus_rate"] = (
    overall_offset_df["current_multi_city_3plus_users"] /
    overall_offset_df["active_user_buckets"].replace(0, np.nan)
)
overall_offset_df["current_mixed_sell_let_rate"] = (
    overall_offset_df["current_mixed_sell_let_users"] /
    overall_offset_df["active_user_buckets"].replace(0, np.nan)
)
overall_offset_df["multi_city_2plus_share_among_city_switch"] = (
    overall_offset_df["city_switch_and_multi_city_2plus"] /
    overall_offset_df["city_switched_users"].replace(0, np.nan)
)
overall_offset_df["multi_city_3plus_share_among_city_switch"] = (
    overall_offset_df["city_switch_and_multi_city_3plus"] /
    overall_offset_df["city_switched_users"].replace(0, np.nan)
)
overall_offset_df["ad_type_switch_share_among_category_switch"] = (
    overall_offset_df["category_switch_and_ad_type_switch"] /
    overall_offset_df["category_switched_users"].replace(0, np.nan)
)
overall_offset_df["mixed_sell_let_share_among_category_switch"] = (
    overall_offset_df["category_switch_and_mixed_sell_let"] /
    overall_offset_df["category_switched_users"].replace(0, np.nan)
)

display(overall_offset_df)
overall_offset_df.to_csv(
    os.path.join(OUTPUT_DIR, "04_overall_offset_switch_diagnosis.csv"),
    index=False,
    encoding="utf-8-sig"
)

# Retention by first-seen ad_type
retention_by_ad_type_df = con.execute("""
WITH maxb AS (
    SELECT MAX(bucket_id) AS max_bucket_id
    FROM bucket_calendar
),
cohort_ad AS (
    SELECT
        COALESCE(cohort_ad_type_mode, 'unknown') AS cohort_ad_type_mode,
        cohort_bucket_id,
        COUNT(*) AS cohort_users
    FROM user_cohort
    GROUP BY
        COALESCE(cohort_ad_type_mode, 'unknown'),
        cohort_bucket_id
),
grid AS (
    SELECT
        ca.cohort_ad_type_mode,
        ca.cohort_bucket_id,
        ca.cohort_users,
        r.bucket_offset
    FROM cohort_ad ca
    CROSS JOIN maxb
    CROSS JOIN LATERAL range(
        0,
        CAST(maxb.max_bucket_id - ca.cohort_bucket_id + 1 AS BIGINT)
    ) AS r(bucket_offset)
),
active AS (
    SELECT
        COALESCE(cohort_ad_type_mode, 'unknown') AS cohort_ad_type_mode,
        cohort_bucket_id,
        bucket_offset,
        COUNT(*) AS active_users
    FROM retained_user_state
    GROUP BY
        COALESCE(cohort_ad_type_mode, 'unknown'),
        cohort_bucket_id,
        bucket_offset
)
SELECT
    g.cohort_ad_type_mode,
    g.bucket_offset,
    '+' || CAST(g.bucket_offset AS VARCHAR) AS offset_label,
    SUM(COALESCE(a.active_users, 0)) AS active_users,
    SUM(g.cohort_users) AS eligible_cohort_users,
    SUM(COALESCE(a.active_users, 0)) * 1.0 / NULLIF(SUM(g.cohort_users), 0)
        AS exact_retention_rate
FROM grid g
LEFT JOIN active a
    ON g.cohort_ad_type_mode = a.cohort_ad_type_mode
    AND g.cohort_bucket_id = a.cohort_bucket_id
    AND g.bucket_offset = a.bucket_offset
GROUP BY
    g.cohort_ad_type_mode,
    g.bucket_offset
ORDER BY
    g.cohort_ad_type_mode,
    g.bucket_offset
""").df()

display(retention_by_ad_type_df)
retention_by_ad_type_df.to_csv(
    os.path.join(OUTPUT_DIR, "05_retention_by_first_seen_ad_type.csv"),
    index=False,
    encoding="utf-8-sig"
)

# Category switch decomposition
category_switch_decomp_df = con.execute("""
SELECT
    bucket_offset,
    offset_label,

    CASE
        WHEN category_switched = 1 AND ad_type_switched = 1
            THEN 'category switch + sell/let switch'
        WHEN category_switched = 1 AND ad_type_switched = 0
            THEN 'category switch only'
        WHEN category_switched = 0 AND ad_type_switched = 1
            THEN 'sell/let switch only'
        ELSE 'no category/ad_type switch'
    END AS switch_decomp,

    COUNT(*) AS active_user_buckets
FROM retained_user_state
GROUP BY
    bucket_offset,
    offset_label,
    CASE
        WHEN category_switched = 1 AND ad_type_switched = 1
            THEN 'category switch + sell/let switch'
        WHEN category_switched = 1 AND ad_type_switched = 0
            THEN 'category switch only'
        WHEN category_switched = 0 AND ad_type_switched = 1
            THEN 'sell/let switch only'
        ELSE 'no category/ad_type switch'
    END
ORDER BY bucket_offset, active_user_buckets DESC
""").df()

category_switch_decomp_df["share_within_offset"] = (
    category_switch_decomp_df["active_user_buckets"] /
    category_switch_decomp_df.groupby("bucket_offset")["active_user_buckets"].transform("sum")
)

display(category_switch_decomp_df)
category_switch_decomp_df.to_csv(
    os.path.join(OUTPUT_DIR, "06_category_switch_decomposition.csv"),
    index=False,
    encoding="utf-8-sig"
)

# City breadth diagnosis
city_breadth_df = con.execute("""
SELECT
    bucket_offset,
    offset_label,

    CASE
        WHEN distinct_cities_seen <= 1 THEN '1 city'
        WHEN distinct_cities_seen = 2 THEN '2 cities'
        WHEN distinct_cities_seen >= 3 THEN '3+ cities'
        ELSE 'unknown'
    END AS city_breadth,

    COUNT(*) AS active_user_buckets,
    SUM(city_switched) AS city_switched_users,
    SUM(category_switched) AS category_switched_users,
    SUM(ad_type_switched) AS ad_type_switched_users
FROM retained_user_state
GROUP BY
    bucket_offset,
    offset_label,
    CASE
        WHEN distinct_cities_seen <= 1 THEN '1 city'
        WHEN distinct_cities_seen = 2 THEN '2 cities'
        WHEN distinct_cities_seen >= 3 THEN '3+ cities'
        ELSE 'unknown'
    END
ORDER BY bucket_offset, city_breadth
""").df()

city_breadth_df["city_switch_rate"] = (
    city_breadth_df["city_switched_users"] /
    city_breadth_df["active_user_buckets"].replace(0, np.nan)
)
city_breadth_df["category_switch_rate"] = (
    city_breadth_df["category_switched_users"] /
    city_breadth_df["active_user_buckets"].replace(0, np.nan)
)
city_breadth_df["ad_type_switch_rate"] = (
    city_breadth_df["ad_type_switched_users"] /
    city_breadth_df["active_user_buckets"].replace(0, np.nan)
)

display(city_breadth_df)
city_breadth_df.to_csv(
    os.path.join(OUTPUT_DIR, "07_switch_by_city_breadth.csv"),
    index=False,
    encoding="utf-8-sig"
)

# Top transitions
top_city_transitions_df = con.execute("""
SELECT
    bucket_offset,
    offset_label,
    cohort_city_mode,
    city_mode AS current_city_mode,
    COUNT(*) AS users
FROM retained_user_state
WHERE city_switched = 1
GROUP BY
    bucket_offset,
    offset_label,
    cohort_city_mode,
    city_mode
ORDER BY bucket_offset, users DESC
""").df()

top_city_transitions_df.to_csv(
    os.path.join(OUTPUT_DIR, "08_top_city_transitions.csv"),
    index=False,
    encoding="utf-8-sig"
)

top_category_transitions_df = con.execute("""
SELECT
    bucket_offset,
    offset_label,
    CAST(cohort_category_mode AS VARCHAR) AS cohort_category_mode,
    CAST(category_mode AS VARCHAR) AS current_category_mode,
    COUNT(*) AS users
FROM retained_user_state
WHERE category_switched = 1
GROUP BY
    bucket_offset,
    offset_label,
    CAST(cohort_category_mode AS VARCHAR),
    CAST(category_mode AS VARCHAR)
ORDER BY bucket_offset, users DESC
""").df()

top_category_transitions_df.to_csv(
    os.path.join(OUTPUT_DIR, "09_top_category_transitions.csv"),
    index=False,
    encoding="utf-8-sig"
)

# Quick key offsets
important_offsets = [1, 2, 3, 4, 6, 8, 10]
quick_df = overall_offset_df[overall_offset_df["bucket_offset"].isin(important_offsets)].copy()

display(quick_df)
quick_df.to_csv(
    os.path.join(OUTPUT_DIR, "10_quick_key_offsets.csv"),
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 7. PLOTS
# ============================================================

def save_plotly(fig, name):
    html_path = os.path.join(OUTPUT_DIR, f"{name}.html")
    fig.write_html(html_path)
    print("[SAVE HTML]", html_path)

    try:
        png_path = os.path.join(OUTPUT_DIR, f"{name}.png")
        fig.write_image(png_path, scale=2)
        print("[SAVE PNG ]", png_path)
    except Exception as e:
        print("[SKIP PNG]", name, str(e)[:100])


# 7.1 Overall switch
plot_df = overall_offset_df.melt(
    id_vars=["bucket_offset", "offset_label"],
    value_vars=[
        "city_switch_rate",
        "category_switch_rate",
        "ad_type_switch_rate",
        "current_multi_city_2plus_rate",
        "current_multi_city_3plus_rate",
        "current_mixed_sell_let_rate",
    ],
    var_name="metric",
    value_name="rate"
)

name_map = {
    "city_switch_rate": "Dominant city switched",
    "category_switch_rate": "Dominant category switched",
    "ad_type_switch_rate": "Dominant sell/let switched",
    "current_multi_city_2plus_rate": "Current bucket: 2+ cities",
    "current_multi_city_3plus_rate": "Current bucket: 3+ cities",
    "current_mixed_sell_let_rate": "Current bucket: mixed sell+let",
}
plot_df["metric"] = plot_df["metric"].map(name_map)

fig = px.line(
    plot_df,
    x="offset_label",
    y="rate",
    color="metric",
    markers=True,
    title="Switch & Multi-city Rate Among Retained Users",
    hover_data={"bucket_offset": True, "rate": ":.2%"}
)
fig.update_layout(
    xaxis_title="Bucket offset",
    yaxis_title="Rate among retained active users",
    yaxis_tickformat=".0%",
    height=600,
    width=1150
)
fig.show()
save_plotly(fig, "01_switch_and_multicity_overall")

# 7.2 City switch diagnosis
plot_df = overall_offset_df.melt(
    id_vars=["bucket_offset", "offset_label"],
    value_vars=[
        "current_multi_city_2plus_rate",
        "multi_city_2plus_share_among_city_switch",
        "current_multi_city_3plus_rate",
        "multi_city_3plus_share_among_city_switch",
    ],
    var_name="metric",
    value_name="rate"
)

name_map = {
    "current_multi_city_2plus_rate": "All retained: 2+ cities",
    "multi_city_2plus_share_among_city_switch": "City switchers: 2+ cities",
    "current_multi_city_3plus_rate": "All retained: 3+ cities",
    "multi_city_3plus_share_among_city_switch": "City switchers: 3+ cities",
}
plot_df["metric"] = plot_df["metric"].map(name_map)

fig = px.line(
    plot_df,
    x="offset_label",
    y="rate",
    color="metric",
    markers=True,
    title="Does City Switching Come From Multi-city Users?",
    hover_data={"bucket_offset": True, "rate": ":.2%"}
)
fig.update_layout(
    xaxis_title="Bucket offset",
    yaxis_title="Rate",
    yaxis_tickformat=".0%",
    height=600,
    width=1150
)
fig.show()
save_plotly(fig, "02_city_switch_multicity_diagnosis")

# 7.3 Retention by let/sell
fig = px.line(
    retention_by_ad_type_df,
    x="offset_label",
    y="exact_retention_rate",
    color="cohort_ad_type_mode",
    markers=True,
    title="Exact Retention by First-seen Dominant ad_type: let vs sell",
    hover_data={
        "bucket_offset": True,
        "active_users": ":,",
        "eligible_cohort_users": ":,",
        "exact_retention_rate": ":.2%",
    }
)
fig.update_layout(
    xaxis_title="Bucket offset",
    yaxis_title="Exact retention rate",
    yaxis_tickformat=".0%",
    height=550,
    width=1100
)
fig.show()
save_plotly(fig, "03_retention_by_first_seen_ad_type")

# 7.4 Category decomposition
fig = px.bar(
    category_switch_decomp_df,
    x="offset_label",
    y="share_within_offset",
    color="switch_decomp",
    title="Category Switch Decomposition: Is It Actually Sell/Let Switch?",
    hover_data={
        "bucket_offset": True,
        "active_user_buckets": ":,",
        "share_within_offset": ":.2%",
    }
)
fig.update_layout(
    xaxis_title="Bucket offset",
    yaxis_title="Share among retained active users",
    yaxis_tickformat=".0%",
    barmode="stack",
    height=600,
    width=1150
)
fig.show()
save_plotly(fig, "04_category_switch_decomposition")

# 7.5 City breadth
plot_df = city_breadth_df.melt(
    id_vars=["bucket_offset", "offset_label", "city_breadth"],
    value_vars=[
        "city_switch_rate",
        "category_switch_rate",
        "ad_type_switch_rate",
    ],
    var_name="metric",
    value_name="rate"
)

name_map = {
    "city_switch_rate": "City switch rate",
    "category_switch_rate": "Category switch rate",
    "ad_type_switch_rate": "Sell/let switch rate",
}
plot_df["metric"] = plot_df["metric"].map(name_map)

fig = px.line(
    plot_df,
    x="offset_label",
    y="rate",
    color="city_breadth",
    line_dash="metric",
    markers=True,
    title="Switch Rates by Current Bucket City Breadth",
    hover_data={"bucket_offset": True, "rate": ":.2%"}
)
fig.update_layout(
    xaxis_title="Bucket offset",
    yaxis_title="Switch rate",
    yaxis_tickformat=".0%",
    height=650,
    width=1200
)
fig.show()
save_plotly(fig, "05_switch_by_city_breadth")

# ============================================================
# 8. DONE
# ============================================================

print("\nDONE.")
print("Output folder:", OUTPUT_DIR)

print("""
CÁCH ĐỌC NHANH:

1) 10_quick_key_offsets.csv
   - Nhìn các offset +1, +2, +4, +6, +8, +10.
   - city_switch_rate: bao nhiêu retained user đổi dominant city.
   - category_switch_rate: bao nhiêu retained user đổi dominant category.
   - ad_type_switch_rate: bao nhiêu retained user đổi dominant sell/let.

2) 02_city_switch_multicity_diagnosis.html
   - Nếu "City switchers: 2+ cities" cao hơn nhiều "All retained: 2+ cities":
     city switch đang bị nhóm xem nhiều city kéo lên.
   - Nếu không cao hơn nhiều:
     city switch có thể là đổi khu vực thật.

3) 03_retention_by_first_seen_ad_type.html
   - So let vs sell.
   - Nếu sell giữ chân lâu hơn let:
     mua nhà có journey dài hơn thuê.
   - Nếu let rớt nhanh:
     thuê có thể là nhu cầu ngắn hạn, quyết định nhanh hơn.

4) 04_category_switch_decomposition.html
   - Nếu phần "category switch only" lớn:
     user đổi phân khúc nhưng vẫn cùng thuê hoặc cùng mua.
   - Nếu "category switch + sell/let switch" lớn:
     user đổi cả intent lớn: thuê <-> mua.

5) 05_switch_by_city_breadth.html
   - Nhóm 3+ cities mà switch rate cao bất thường:
     có thể là broad search / professional browsing / cò-like behavior.
""")

print("Saved CSV/HTML:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".csv") or f.endswith(".html"):
        print("-", f)

con.close()

EVENTS_PATH: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events
DIM_PATH   : /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/dim_listing
EVENT_FILES: 500
OUTPUT_DIR : /kaggle/working/retention_lowdisk_sell_let_multicity

[1/8] Build dim_lite...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,items,sell_items,let_items,unknown_ad_type_items
0,3107114,3107114,1620385.0,1486729.0,0.0



[2/8] Per-file aggregation...
[1/500] datathon_fact_user_events-000000000000.parquet
[2/500] datathon_fact_user_events-000000000001.parquet
[3/500] datathon_fact_user_events-000000000002.parquet
[4/500] datathon_fact_user_events-000000000003.parquet
[5/500] datathon_fact_user_events-000000000004.parquet
[6/500] datathon_fact_user_events-000000000005.parquet
[7/500] datathon_fact_user_events-000000000006.parquet
[8/500] datathon_fact_user_events-000000000007.parquet
[9/500] datathon_fact_user_events-000000000008.parquet
[10/500] datathon_fact_user_events-000000000009.parquet
[11/500] datathon_fact_user_events-000000000010.parquet
[12/500] datathon_fact_user_events-000000000011.parquet
[13/500] datathon_fact_user_events-000000000012.parquet
[14/500] datathon_fact_user_events-000000000013.parquet
[15/500] datathon_fact_user_events-000000000014.parquet
[16/500] datathon_fact_user_events-000000000015.parquet
[17/500] datathon_fact_user_events-000000000016.parquet
[18/500] datathon_fact_use

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

[SAVE HTML] /kaggle/working/retention_lowdisk_sell_let_multicity/01_switch_and_multicity_overall.html
[SKIP PNG] 01_switch_and_multicity_overall 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using 


[SAVE HTML] /kaggle/working/retention_lowdisk_sell_let_multicity/02_city_switch_multicity_diagnosis.html
[SKIP PNG] 02_city_switch_multicity_diagnosis 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using 


[SAVE HTML] /kaggle/working/retention_lowdisk_sell_let_multicity/03_retention_by_first_seen_ad_type.html
[SKIP PNG] 03_retention_by_first_seen_ad_type 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using 


[SAVE HTML] /kaggle/working/retention_lowdisk_sell_let_multicity/04_category_switch_decomposition.html
[SKIP PNG] 04_category_switch_decomposition 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using 


[SAVE HTML] /kaggle/working/retention_lowdisk_sell_let_multicity/05_switch_by_city_breadth.html
[SKIP PNG] 05_switch_by_city_breadth 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using 

DONE.
Output folder: /kaggle/working/retention_lowdisk_sell_let_multicity

CÁCH ĐỌC NHANH:

1) 10_quick_key_offsets.csv
   - Nhìn các offset +1, +2, +4, +6, +8, +10.
   - city_switch_rate: bao nhiêu retained user đổi dominant city.
   - category_switch_rate: bao nhiêu retained user đổi dominant category.
   - ad_type_switch_rate: bao nhiêu retained user đổi dominant sell/let.

2) 02_city_switch_multicity_diagnosis.html
   - Nếu "City switchers: 2+ cities" cao hơn nhiều "All retained: 2+ cities":
     city switch đang bị nhóm xem nhiều city kéo lên.
   - Nếu không cao hơn nhiều:
     city switch có thể là đổi khu vực thật.

3) 03_retention_by_first_seen_ad_type.html
   - So let vs sell.
   - Nếu sell giữ chân lâu hơn let:
     mua nhà có journey dài hơn th